# Classificacao de Imagens com timm MobileNetV3

Este notebook classifica as imagens da pasta `images` usando o modelo `mobilenetv3_small_100.lamb_in1k` da biblioteca `timm`.

## Padrao de documentacao deste notebook
- Toda nova etapa deve ter **titulo** e **descricao** em celula markdown.
- Toda celula de codigo deve incluir **comentarios curtos** explicando a intencao.

## 1) Imports e configuracao inicial

Aqui importamos bibliotecas, definimos dispositivo de inferencia e configuramos caminhos de entrada.

In [ ]:
# Importa bibliotecas basicas para sistema de arquivos e exibicao de resultados.
from pathlib import Path
from pprint import pprint

# Importa PyTorch e utilitarios de modelo/transforms da biblioteca timm.
import torch
from PIL import Image
import timm
from timm.data import resolve_model_data_config, create_transform, ImageNetInfo, infer_imagenet_subset

# Define caminho da pasta de imagens e dispositivo de execucao.
images_dir = Path("images")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Exibe informacoes uteis de ambiente para depuracao.
print(f"timm: {timm.__version__}")
print(f"torch: {torch.__version__}")
print(f"device: {device}")

## 2) Carregamento do modelo e pre-processamento

Nesta etapa carregamos o modelo pre-treinado, montamos o transform correto e recuperamos os nomes das classes.

In [ ]:
# Define identificador do modelo que sera utilizado para classificacao.
model_name = "mobilenetv3_small_100.lamb_in1k"

# Carrega o modelo com pesos pre-treinados e coloca em modo de inferencia.
model = timm.create_model(model_name, pretrained=True)
model.to(device)
model.eval()

# Resolve configuracao de dados do modelo e cria transform compativel.
data_config = resolve_model_data_config(model)
transform = create_transform(**data_config, is_training=False)

# Recupera os nomes das classes do subset ImageNet associado ao modelo.
imagenet_info = ImageNetInfo(infer_imagenet_subset(model))
label_names = imagenet_info.label_names()

print(f"Modelo carregado: {model_name}")
print(f"Numero de classes: {len(label_names)}")

## 3) Classificacao das imagens da pasta `images`

A celula abaixo percorre todas as imagens suportadas, roda inferencia e salva Top-1 e Top-5 para cada arquivo.

In [ ]:
# Define extensoes de imagem permitidas para classificacao.
valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp"}

# Lista arquivos validos na pasta de entrada.
image_paths = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in valid_ext])
if not image_paths:
    raise FileNotFoundError("Nenhuma imagem encontrada na pasta 'images'.")

# Acumula resultados estruturados de classificacao por imagem.
results = []

with torch.inference_mode():
    for image_path in image_paths:
        # Abre imagem em RGB para garantir compatibilidade com o transform.
        image = Image.open(image_path).convert("RGB")

        # Aplica pre-processamento e adiciona dimensao de batch.
        input_tensor = transform(image).unsqueeze(0).to(device)

        # Executa inferencia e converte logits em probabilidades.
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1)

        # Seleciona as 5 classes com maior probabilidade.
        top_probs, top_indices = torch.topk(probs, k=5, dim=1)
        top_probs = top_probs[0].cpu().tolist()
        top_indices = top_indices[0].cpu().tolist()

        # Monta lista estruturada de previsoes Top-5.
        top5 = []
        for idx, prob in zip(top_indices, top_probs):
            top5.append({
                "class_id": int(idx),
                "label": label_names[idx],
                "probability": float(prob),
            })

        # Guarda resultado final da imagem (Top-1 + Top-5).
        results.append({
            "image": image_path.name,
            "top1": top5[0],
            "top5": top5,
        })

print(f"Total de imagens classificadas: {len(results)}")

## 4) Visualizacao dos resultados

Esta etapa exibe os resultados de forma estruturada para facilitar validacao e comparacao.

In [ ]:
# Imprime um resumo simples por imagem com classe Top-1 e probabilidade.
for item in results:
    top1 = item["top1"]
    print(f"{item['image']}: {top1['label']} ({top1['probability']:.4f})")

print("\nResultados completos (Top-5 por imagem):")

# Exibe estrutura completa para inspecao detalhada.
pprint(results, sort_dicts=False)